In [13]:
%pip -q install openai-whisper jiwer pandas numpy
import subprocess, shutil
if not shutil.which("ffmpeg"):
    print("WARNING: ffmpeg not found. Install it: brew install ffmpeg (macOS) or apt install ffmpeg (Linux)")
else:
    print("ffmpeg found.")
print("Dependencies ready.")

Note: you may need to restart the kernel to use updated packages.
ffmpeg found.
Dependencies ready.


In [14]:
import os
import re
import time
import wave
import struct
import numpy as np
import whisper
import pandas as pd
from IPython.display import display

AUDIO_DIR = "./audio/selected_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

_model_cache = {}

def get_model(name="base"):
    if name not in _model_cache:
        print(f"Loading Whisper model '{name}'...")
        _model_cache[name] = whisper.load_model(name)
    return _model_cache[name]

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def read_csv_auto(path):
    for enc in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1")

print("Imports and helpers ready.")

Imports and helpers ready.


In [15]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    clean = normalize_text(transcript)
    words = clean.split()

    state_intents = INTENT_CONFIG.get(state, {})

    for intent_name, groups in state_intents.items():
        for group in groups:
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown"

In [16]:
TEMPLATE_WAV_PATH = "./utils/template_audio/audioPromptPrefix.wav"
APPENDED_AUDIO_DIR = "./appended_audio"
TEMPLATE_VOLUME = 0.1

os.makedirs(APPENDED_AUDIO_DIR, exist_ok=True)

def read_wav_samples(wav_path):
    with wave.open(wav_path, "rb") as wf:
        params = wf.getparams()
        raw = wf.readframes(wf.getnframes())
    samples = np.frombuffer(raw, dtype=np.int16)
    return samples, params

def compute_rms(samples):
    if len(samples) == 0:
        return 0.0
    return np.sqrt(np.mean(samples.astype(np.float64) ** 2))

def apply_gain(samples, gain):
    amplified = np.round(samples.astype(np.float64) * gain)
    return np.clip(amplified, -32768, 32767).astype(np.int16)

def write_wav(path, samples, params):
    with wave.open(path, "wb") as wf:
        wf.setnchannels(params.nchannels)
        wf.setsampwidth(params.sampwidth)
        wf.setframerate(params.framerate)
        wf.writeframes(samples.tobytes())

def append_template_to_audio(mic_wav_path, template_wav_path=TEMPLATE_WAV_PATH):
    mic_name = os.path.splitext(os.path.basename(mic_wav_path))[0]
    template_name = os.path.splitext(os.path.basename(template_wav_path))[0]
    output_name = f"{mic_name}-appended-with-{template_name}.wav"
    output_path = os.path.join(APPENDED_AUDIO_DIR, output_name)

    print(f"  [wavAppender] --- START ---")
    print(f"  [wavAppender] Mic file:      {os.path.basename(mic_wav_path)}")
    print(f"  [wavAppender] Template file:  {os.path.basename(template_wav_path)}")

    template_samples, template_params = read_wav_samples(template_wav_path)
    mic_samples, mic_params = read_wav_samples(mic_wav_path)

    print(f"  [wavAppender] Mic:      {mic_params.framerate}Hz, {mic_params.nchannels}ch, {len(mic_samples)} samples ({len(mic_samples)/mic_params.framerate:.2f}s)")
    print(f"  [wavAppender] Template: {template_params.framerate}Hz, {template_params.nchannels}ch, {len(template_samples)} samples ({len(template_samples)/template_params.framerate:.2f}s)")

    if mic_params.framerate != template_params.framerate:
        print(f"  [wavAppender] ERROR: Sample rate mismatch (template={template_params.framerate}, mic={mic_params.framerate}). Skipping.")
        print(f"  [wavAppender] --- END (FAILED) ---")
        return None

    template_rms = compute_rms(template_samples)
    mic_rms = compute_rms(mic_samples)
    template_gain = min(mic_rms / template_rms, 1.0) * TEMPLATE_VOLUME if template_rms > 0 and mic_rms > 0 else 1.0

    print(f"  [wavAppender] Template RMS: {template_rms:.2f} | Mic RMS: {mic_rms:.2f} | Gain: {template_gain:.4f}")

    attenuated_template = apply_gain(template_samples, template_gain)
    attenuated_rms = compute_rms(attenuated_template)
    print(f"  [wavAppender] After attenuation -> Template RMS: {attenuated_rms:.2f} | Mic RMS: {mic_rms:.2f}")

    appended = np.concatenate([attenuated_template, mic_samples])
    total_duration = len(appended) / template_params.framerate

    write_wav(output_path, appended, template_params)
    file_size = os.path.getsize(output_path)
    print(f"  [wavAppender] Output:   {output_name} ({total_duration:.2f}s, {file_size} bytes)")
    print(f"  [wavAppender] Saved to: {output_path}")
    print(f"  [wavAppender] --- END (OK) ---")
    return output_path

print("WAV appender ready.")
print(f"Appended audio output dir: {os.path.abspath(APPENDED_AUDIO_DIR)}")
if os.path.exists(TEMPLATE_WAV_PATH):
    t_samples, t_params = read_wav_samples(TEMPLATE_WAV_PATH)
    print(f"Template: {t_params.framerate}Hz, {t_params.nchannels}ch, {len(t_samples)} samples ({len(t_samples)/t_params.framerate:.2f}s)")
else:
    print(f"WARNING: Template not found at {TEMPLATE_WAV_PATH}")

WAV appender ready.
Appended audio output dir: /Users/fandikaikhsan/Documents/norilla/whisper-pipeline/appended_audio
Template: 16000Hz, 1ch, 22988 samples (1.44s)


In [17]:
INPUT_CSV = "./reason_prediction_audio_log.csv"
WHISPER_MODEL_BATCH = "base"
BATCH_STATE = "default"
USE_APPENDING = True  # Set to False to skip template prepending

KEYWORDS = {
    "reason": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric"
    ],
    "decision": [
        "right", "two", "left", "one", "both", "together", "same time",
    ],
    "default": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric",
        "right", "two", "left", "one", "both", "together", "same time",
    ],
}

def run_batch_pipeline_with_appending(csv_path, model_name, state, use_appending=True):
    if not os.path.exists(csv_path):
        print(f"[pipeline] Error: CSV not found at {csv_path}")
        return

    if use_appending and not os.path.exists(TEMPLATE_WAV_PATH):
        print(f"[pipeline] Error: Template WAV not found at {TEMPLATE_WAV_PATH}")
        return

    df = read_csv_auto(csv_path)

    suffix = "appended" if use_appending else "no-append"
    transcript_col = f"re-run transcription [{suffix}]"
    intent_col = f"re-run intent [{state}] [{suffix}]"
    timing_col = f"re-run processing time [ms] [{suffix}]"

    for col in [transcript_col, intent_col, timing_col]:
        if col not in df.columns:
            df[col] = ""

    model = get_model(model_name)
    processed_count = 0
    skipped_count = 0
    error_count = 0

    keywords = KEYWORDS.get(state, KEYWORDS["default"])
    batch_prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(keywords)

    print(f"[pipeline] ========================================")
    print(f"[pipeline] CSV:           {csv_path}")
    print(f"[pipeline] Whisper model: {model_name}")
    print(f"[pipeline] State:         {state}")
    print(f"[pipeline] Appending:     {use_appending}")
    if use_appending:
        print(f"[pipeline] Template:      {TEMPLATE_WAV_PATH}")
        print(f"[pipeline] Output dir:    {APPENDED_AUDIO_DIR}")
    print(f"[pipeline] Prompt:        {batch_prompt}")
    print(f"[pipeline] Total rows:    {len(df)}")
    print(f"[pipeline] ========================================\n")

    for idx, row in df.iterrows():
        if str(row.get(intent_col, '')).strip() not in ["", "nan"]:
            continue

        file_name = str(row['fileName']).strip()
        wav_path = os.path.join(AUDIO_DIR, file_name)

        if not os.path.exists(wav_path):
            skipped_count += 1
            print(f"[pipeline] Row {idx}: SKIP (file not found: {file_name})")
            continue

        existing_text = str(row.get(transcript_col, '')).strip()
        if existing_text in ["", "nan"]:
            try:
                transcribe_path = wav_path
                if use_appending:
                    print(f"\n[pipeline] Row {idx}: Appending template to {file_name}")
                    appended_path = append_template_to_audio(wav_path)
                    if appended_path is None:
                        print(f"[pipeline] Row {idx}: SKIP (appending failed)")
                        skipped_count += 1
                        continue
                    transcribe_path = appended_path

                print(f"[pipeline] Row {idx}: Transcribing {os.path.basename(transcribe_path)}...")
                start_time = time.time()
                result = model.transcribe(transcribe_path, initial_prompt=batch_prompt, language="en")
                elapsed_ms = (time.time() - start_time) * 1000

                new_text = result["text"].strip()
                df.at[idx, transcript_col] = new_text
                df.at[idx, timing_col] = round(elapsed_ms, 2)
                print(f"[pipeline] Row {idx}: Transcription: \"{new_text}\" ({elapsed_ms:.0f}ms)")
            except Exception as e:
                print(f"[pipeline] Row {idx}: ERROR: {file_name} — {e}")
                df.at[idx, transcript_col] = "ERROR"
                error_count += 1
                continue
        else:
            new_text = existing_text
            print(f"[pipeline] Row {idx}: Using existing transcription: \"{new_text}\"")

        new_intent = determine_intent(new_text, state)
        df.at[idx, intent_col] = new_intent
        processed_count += 1
        print(f"[pipeline] Row {idx}: Intent: {new_intent}")

    df.to_csv(csv_path, index=False)
    print(f"\n[pipeline] ========================================")
    print(f"[pipeline] DONE!")
    print(f"[pipeline] Processed: {processed_count} | Skipped: {skipped_count} | Errors: {error_count}")
    if use_appending:
        saved_files = [f for f in os.listdir(APPENDED_AUDIO_DIR) if f.endswith(".wav")]
        print(f"[pipeline] Appended audio files saved: {len(saved_files)} in {APPENDED_AUDIO_DIR}/")
    print(f"[pipeline] ========================================")
    display(df.head(10))

run_batch_pipeline_with_appending(INPUT_CSV, WHISPER_MODEL_BATCH, BATCH_STATE, USE_APPENDING)

Loading Whisper model 'base'...
[pipeline] ========================================
[pipeline] CSV:           ./reason_prediction_audio_log.csv
[pipeline] Whisper model: base
[pipeline] State:         default
[pipeline] Appending:     True
[pipeline] Template:      ./utils/template_audio/audioPromptPrefix.wav
[pipeline] Output dir:    ./appended_audio
[pipeline] Prompt:        Try to focus specifically on these words as you listen to the audio: tall, taller, big, high, higher, weight, weighs, top, heavy, heavier, thin, thinner, small, smaller, symmetric, symmetrical, both sides, asymmetrical, asymmetric, right, two, left, one, both, together, same time
[pipeline] Total rows:    124
[pipeline] ========================================

[pipeline] Row 0: SKIP (file not found: 2026-6-14--10-14--3.wav)
[pipeline] Row 1: SKIP (file not found: 2026-6-14--10-14--4.wav)
[pipeline] Row 2: SKIP (file not found: 2026-6-14--10-14--5.wav)
[pipeline] Row 3: SKIP (file not found: 2026-6-14--10-18--2.w

/usr/local/anaconda3/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[pipeline] Row 20: Transcription: "the answer is..." (1351ms)
[pipeline] Row 20: Intent: default.unknown
[pipeline] Row 21: SKIP (file not found: 2026-6-14--11-0--8.wav)
[pipeline] Row 22: SKIP (file not found: 2026-6-14--11-1--9.wav)
[pipeline] Row 23: SKIP (file not found: 2026-6-14--11-3--1.wav)
[pipeline] Row 24: SKIP (file not found: 2026-6-14--11-4--5.wav)
[pipeline] Row 25: SKIP (file not found: 2026-6-14--11-5--6.wav)
[pipeline] Row 26: SKIP (file not found: 2026-6-14--11-5--7.wav)
[pipeline] Row 27: SKIP (file not found: 2026-6-14--11-11--2.wav)
[pipeline] Row 28: SKIP (file not found: 2026-6-14--11-13--5.wav)
[pipeline] Row 29: SKIP (file not found: 2026-6-14--11-13--6.wav)
[pipeline] Row 30: SKIP (file not found: 2026-6-14--11-13--7.wav)
[pipeline] Row 31: SKIP (file not found: 2026-6-14--11-19--5.wav)
[pipeline] Row 32: SKIP (file not found: 2026-6-14--11-19--6.wav)
[pipeline] Row 33: SKIP (file not found: 2026-6-14--11-19--7.wav)
[pipeline] Row 34: SKIP (file not found: 20

,date,time,duration_ms,filePath,fileName,fileSize_bytes,format,sampleRate,transcription,confidence,filePath.1,Process Time,Intent,Manual Transcription,Speaker,re-run transcription [appended],re-run intent [default] [appended],re-run processing time [ms] [appended]
0,6/14/2026,10:14:10,6094,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-14--3.wav,153646,WAV,16000,cut up a cut up in our base.,0.4776,C:\Users\noril\Downloads\processing-3.5.4\data...,1.385048,tower.reason.thinnerbase,Because it has a thinner base,Norilla,,,
1,6/14/2026,10:14:24,6009,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-14--4.wav,153646,WAV,16000,with the head of inner base.,0.5231,C:\Users\noril\Downloads\processing-3.5.4\data...,1.317288,tower.reason.thinnerbase,Because it had a thinner base,Norilla,,,
2,6/14/2026,10:14:37,6060,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-14--5.wav,153646,WAV,16000,OVER 5 SECONDS,0.0000,C:\Users\noril\Downloads\processing-3.5.4\data...,0.000000,default.noaudio,NO SPOKEN WORD,NaN,,,
3,6/14/2026,10:18:07,6039,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-18--2.wav,163886,WAV,16000,"we'll come back and have more weight, just one...",0.4027,C:\Users\noril\Downloads\processing-3.5.4\data...,1.601501,tower.reason.weight,Because it has more weight on top than bottom,Norilla,,,
4,6/14/2026,10:18:45,6004,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-18--4.wav,163886,WAV,16000,because it has a thinner base.,0.5083,C:\Users\noril\Downloads\processing-3.5.4\data...,1.447961,tower.reason.thinnerbase,Because it has a thinner base,Norilla,,,
5,6/14/2026,10:22:44,6057,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-22--6.wav,163886,WAV,16000,in that case we have to make it a little bit s...,0.4194,C:\Users\noril\Downloads\processing-3.5.4\data...,1.845466,tower.reason.tall,Yeah. In that case we have to make it cause i...,Norilla,,,
6,6/14/2026,10:23:55,6027,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-23--8.wav,163886,WAV,16000,it's easy because that's how we're once taller...,0.4802,C:\Users\noril\Downloads\processing-3.5.4\data...,1.646755,tower.reason.tall,Because that tower is taller than the other tower,Norilla,,,
7,6/14/2026,10:24:33,6091,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-24--10.wav,163886,WAV,16000,in color that has a smaller base.,0.4298,C:\Users\noril\Downloads\processing-3.5.4\data...,1.355473,tower.reason.thinnerbase,Because it has a smaller base,Norilla,,,
8,6/14/2026,10:25:15,6100,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-25--12.wav,163886,WAV,16000,and not the same on both sides.,0.4395,C:\Users\noril\Downloads\processing-3.5.4\data...,1.760956,tower.reason.notsymmetrical,Um It's not the same on both sides,Norilla,,,
9,6/14/2026,10:26:55,6042,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-26--15.wav,153646,WAV,16000,"Hello, the bottom is the body straight.",0.3684,C:\Users\noril\Downloads\processing-3.5.4\data...,2.693729,NaN,no the bottom is,User,,,


In [18]:
def run_intent_classification_appended(csv_path, use_appending=True):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        return

    df = read_csv_auto(csv_path)

    suffix = "appended" if use_appending else "no-append"
    transcript_col = f"re-run transcription [{suffix}]"
    decision_col = f"re-run intent [decision] [{suffix}]"
    reason_col = f"re-run intent [reason] [{suffix}]"

    if transcript_col not in df.columns:
        print(f"No '{transcript_col}' column found. Run the batch pipeline first.")
        return

    for col in [decision_col, reason_col]:
        if col not in df.columns:
            df[col] = ""

    classified = 0
    for idx, row in df.iterrows():
        transcript = str(row.get(transcript_col, '')).strip()
        if transcript in ["", "nan", "ERROR"]:
            continue

        df.at[idx, decision_col] = determine_intent(transcript, "decision")
        df.at[idx, reason_col] = determine_intent(transcript, "reason")
        classified += 1

    df.to_csv(csv_path, index=False)
    print(f"Classified {classified} rows for both decision and reason intents.")
    display(df[[transcript_col, decision_col, reason_col]].head(20))

run_intent_classification_appended(INPUT_CSV, USE_APPENDING)

Classified 1 rows for both decision and reason intents.


,re-run transcription [appended],re-run intent [decision] [appended],re-run intent [reason] [appended]
0,NaN,,
1,NaN,,
2,NaN,,
3,NaN,,
4,NaN,,
5,NaN,,
6,NaN,,
7,NaN,,
8,NaN,,
9,NaN,,
